# Inheco ODTC

PCR-oriented thermocycler: **4–99 °C**, heated lid, motorized door, **96** or **384** wells. Ethernet via **SiLA 1 (SOAP)** — you need the device IP.

[Hardware](https://www.inheco.com/odtc.html).

## Architecture (v1b1)

- **`ODTC`** — `Device` + deck `Resource`; **`setup()`** / **`stop()`** run the SiLA session and wire up the pieces below.
- **`odtc.driver`** — SiLA transport to the instrument (all commands share this).
- **`odtc.tc`** — **`Thermocycler`**: temperatures, protocols, **`run_protocol`**, **`wait_for_completion`**.
- **`odtc.door`** — **`LoadingTray`**: motorized door and plate access for arms.
- **`Protocol` / `Stage` / `Step` / `Ramp`** — abstract run model. **`ODTCBackendParams`** — ODTC compile options for **`run_protocol()`** / **`ODTCProtocol.from_protocol()`**.


## Setup

In [ ]:
from pylabrobot.inheco.odtc import ODTC, DoorStateUnknownError, FluidQuantity, ODTCBackendParams
from pylabrobot.inheco.odtc.model import ODTCProtocol
from pylabrobot.capabilities.thermocycling import (
    Protocol,
    Stage,
    Step,
    Ramp,
    Overshoot,
    FULL_SPEED,
)

odtc = ODTC(
    odtc_ip="192.168.1.50",  # replace with your ODTC's IP address
    variant=96,                 # 96 or 384
    name="odtc",
)

await odtc.setup()  # connects, resets, and initialises the device

> **Tip:** Thermocycler: `odtc.tc`. Door/plate resource: `odtc.door`. **SiLA** (SOAP) is **`odtc.driver`** (`ODTCDriver` in `driver.py`); `ODTC.setup()` / `stop()` start and stop that connection.


## Door

`LoadingTray`: door control and plate spot for arms.

**State:** firmware does not report door position — PLR tracks it after `open()` / `close()`. Until then, `odtc.door.backend.is_open` raises **`DoorStateUnknownError`**. After reconnect, state is unknown again.


In [ ]:
await odtc.door.open()

In [ ]:
# Check door state (raises DoorStateUnknownError if neither open() nor close() has been called)
try:
    print("Door is open:", odtc.door.backend.is_open)
except DoorStateUnknownError as e:
    print("State unknown:", e)

In [ ]:
await odtc.door.close()

## Temperatures

`request_temperatures()` returns all sensors. `request_block_temperature()` / `request_lid_temperature()` are shortcuts:


In [ ]:
sensors = await odtc.tc.backend.request_temperatures()
print(sensors)

In [ ]:
block_temp = await odtc.tc.request_block_temperature()
lid_temp   = await odtc.tc.request_lid_temperature()
print(f"Block: {block_temp:.1f} °C   Lid: {lid_temp:.1f} °C")

## Holding (pre-method)

`set_block_temperature(t)` uploads a **pre-method** (ramp + hold). Returns when the command is accepted — call **`wait_for_completion()`** to wait until ready. First ramp to a new setpoint can take several minutes.

**Before `run_protocol`:** the block should match your protocol’s **first** `Step.temperature` (this example: **95 °C**). **`run_protocol` does not preheat.** Typical flow: **`set_block_temperature`** → **`wait_for_completion`** → **`run_protocol`**.

**Lid:** align with `Protocol.lid_temperature` (and any per-step overrides). `set_block_temperature` uses backend defaults for the pre-method; use `SetBlockTempParams` if you need explicit lid matching.

**Follow-up method:** if the next method’s first step differs from the current block, preheat again first.

`deactivate_block()` ends the hold.


In [ ]:
# Example hold temperature (not the PCR preheat target — see Running a PCR protocol)
await odtc.tc.set_block_temperature(37.0)

In [ ]:
temp = await odtc.tc.request_block_temperature()
print(f"Block: {temp:.1f} °C")

In [ ]:
await odtc.tc.deactivate_block()

## Running a PCR protocol

Same **`Protocol` → `Stage` → `Step`** model as in [Thermocycling](../../01_material-handling/thermocycling/thermocycling).

| Type | Key fields |
|------|------------|
| `Step` | `temperature`, `hold_seconds`, optional `ramp` |
| `Ramp` | `rate` (°C/s), optional `overshoot` |
| `Stage` | `steps`, `repeats`; optional nested `inner_stages` |
| `Protocol` | `stages`, `name`, `lid_temperature` |

### Example protocol


In [ ]:
pcr_protocol = Protocol(
    name="StandardPCR",
    lid_temperature=105.0,   # default lid temp for all steps
    stages=[
        # Initial denaturation — 5 min at 95 °C
        Stage(
            steps=[Step(temperature=95.0, hold_seconds=300)],
            repeats=1,
        ),
        # PCR cycling — 30 cycles
        Stage(
            steps=[
                Step(temperature=95.0, hold_seconds=30),   # Denature
                Step(temperature=55.0, hold_seconds=30),   # Anneal
                Step(temperature=72.0, hold_seconds=60),   # Extend
            ],
            repeats=30,
        ),
        # Final extension — 10 min at 72 °C
        Stage(
            steps=[Step(temperature=72.0, hold_seconds=600)],
            repeats=1,
        ),
        # Hold at 4 °C — post_heating=True (default) keeps the block here
        # after the method completes. No explicit infinite-hold step needed.
        Stage(
            steps=[Step(temperature=4.0, hold_seconds=30)],
            repeats=1,
        ),
    ],
)

### Volume, overshoot, backend params

**Volume → `FluidQuantity`:** `volume_ul` (max µL/well) maps **1–29** / **30–74** / **75–100** to `UL_10_TO_29` / `UL_30_TO_74` / `UL_75_TO_100`. Default when unset: **30–74**. **`backend_params.fluid_quantity`** overrides `volume_ul`.

**Overshoot:** **`apply_overshoot=True`** (default) adds auto overshoot when a step has no `Ramp.overshoot`. **`False`** turns that off; **explicit** `Ramp(..., overshoot=Overshoot(...))` always applies. Same flag via `ODTCProtocol.from_protocol(..., params=ODTCBackendParams(...))`.

```python
await odtc.tc.run_protocol(pcr_protocol, volume_ul=50.0)
# dynamic_pre_method_duration only on run_protocol(), not ODTCBackendParams:
# await odtc.tc.run_protocol(pcr_protocol, volume_ul=50.0, dynamic_pre_method_duration=True)
```

Variant is chosen at construction (`ODTC(variant=96|384)`).

After `run_protocol`, **`wait_for_completion(timeout=..., report_interval=...)`** logs INFO progress every `report_interval` seconds (default **300**; **`0`** disables).


In [ ]:
# Option A: pass volume_ul directly — fluid_quantity auto-derived
params_a = None  # no backend_params needed; just pass volume_ul to run_protocol

# Option B: explicit fluid_quantity in ODTCBackendParams (overrides volume_ul)
params_b = ODTCBackendParams(
    fluid_quantity=FluidQuantity.UL_30_TO_74,  # 30–74 µL samples
    post_heating=True,
)

### Run: preheat → run → wait


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

# 1) Preheat to the first step temperature (StandardPCR starts at 95 C)
await odtc.tc.set_block_temperature(95.0)
await odtc.tc.wait_for_completion()

# 2) Main method (volume -> FluidQuantity; see markdown above for apply_overshoot / params)
await odtc.tc.run_protocol(pcr_protocol, volume_ul=50.0)
# await odtc.tc.run_protocol(pcr_protocol, backend_params=params_b)  # alternative

# 3) Block until the PCR finishes; periodic INFO lines (elapsed / ~total) every report_interval seconds
await odtc.tc.wait_for_completion(report_interval=120.0)


### Stopping a protocol

In [ ]:
await odtc.tc.stop_protocol()

## Custom ramp rates

`Ramp(rate=...)` in °C/s. **`Ramp()`** / **`FULL_SPEED`** — hardware max. Examples (96-well max heat/cool often **4.4** / **2.2** °C/s).


In [ ]:
precise_protocol = Protocol(
    name="PreciseRamps",
    stages=[
        Stage(
            steps=[
                Step(temperature=95.0, hold_seconds=30, ramp=Ramp(rate=4.4)),  # max heating
                Step(temperature=55.0, hold_seconds=30, ramp=Ramp(rate=2.2)),  # max cooling
                Step(temperature=72.0, hold_seconds=60, ramp=Ramp(rate=1.5)),  # gentle
            ],
            repeats=10,
        ),
    ],
)

await odtc.tc.run_protocol(
    precise_protocol,
    backend_params=ODTCBackendParams(fluid_quantity=FluidQuantity.UL_10_TO_29),
)

## Per-step lid temperature

Default: `Protocol.lid_temperature`. Override: `Step.lid_temperature`. (Heated **cover**, not the motorized `door`.)


In [ ]:
lid_protocol = Protocol(
    name="PerStepLid",
    lid_temperature=105.0,   # default for all steps
    stages=[
        Stage(
            steps=[
                Step(temperature=95.0, hold_seconds=30),                         # lid = 105 °C (default)
                Step(temperature=55.0, hold_seconds=30, lid_temperature=100.0),  # override
                Step(temperature=72.0, hold_seconds=60, lid_temperature=110.0),  # override
            ],
            repeats=5,
        ),
    ],
)

## Stored (named) protocols

Compile with `ODTCProtocol.from_protocol()`, **`upload_protocol()`** once, then **`run_stored_protocol()`** later — survives reset/reconnect until deleted.


In [ ]:
# Compile the protocol to the ODTC's device-native form with a persistent name
odtc_protocol = ODTCProtocol.from_protocol(
    pcr_protocol,
    variant=96,
    params=ODTCBackendParams(
        fluid_quantity=FluidQuantity.UL_30_TO_74,
        name="StandardPCR",          # is_scratch=False: persists on device
        apply_overshoot=True,        # default; set False to disable auto-overshoot
    ),
)
print(odtc_protocol)

# Upload once — survives device Reset and can be run in future sessions
await odtc.tc.backend.upload_protocol(odtc_protocol)

In [ ]:
# List methods currently stored on the device
method_set = await odtc.tc.backend.get_method_set()
print(method_set)

# Retrieve a stored protocol by name (returns None if not found or is a premethod)
stored = await odtc.tc.backend.get_protocol("StandardPCR")
print(stored)

# Run a stored named protocol without re-uploading
# (works even after device Reset or reconnect)
await odtc.tc.backend.run_stored_protocol("StandardPCR")

## 384-well variant

Max heating **5.0** °C/s (vs **4.4** on 96). **`plate_type`** **0** or **2** (96-well: **0** only).


In [ ]:
# odtc_384 = ODTC(odtc_ip="169.254.x.x", variant=384, name="odtc_384")
# await odtc_384.setup()
#
# await odtc_384.tc.run_protocol(
#     pcr_protocol,
#     backend_params=ODTCBackendParams(
#         fluid_quantity=FluidQuantity.UL_75_TO_100,
#         plate_type=0,
#     ),
# )

## Teardown

In [ ]:
await odtc.stop()  # deactivates block, stops driver (closes SiLA)
